#### Category Classifier

##### Importing Packages and Loading Data

In [1]:
# Planscope imports work because the package is installed in editable mode

import pandas as pd

from planscope.category_data import reference_code

df = pd.read_parquet('../data/processed/applications.parquet')

In [2]:
# % of applications with no category, by year submitted: the field arrived around 2021

df['category'].isna().groupby(df['valid_date'].dt.year).mean().mul(100).round(0)

valid_date
2019    93.0
2020    82.0
2021    19.0
2022    13.0
2023     9.0
2024     8.0
2025     9.0
2026    13.0
Name: category, dtype: float64

In [3]:
# The same description can go through different legal routes

text = df['description'].str.lower().str.rstrip('.')
df.loc[text == 'single storey rear extension', 'category'].value_counts(normalize=True).round(2)

category
householder           0.82
lawful_development    0.09
full_planning         0.06
prior_approval        0.03
Name: proportion, dtype: float64

In [4]:
# Ceiling for any text-only model: how often a repeated description's most common category is right

lab = df[df['category'].notna() & df['description'].notna()].assign(text=text)
repeated = lab[lab.duplicated('text', keep=False)]
repeated.groupby('text')['category'].agg(lambda s: s.value_counts().iloc[0]).sum() / len(repeated)

np.float64(0.9118746262706797)

In [6]:
# Council reference codes are reliable labels, e.g. Barnet's HSE

code = reference_code(df['lpa_app_no'])
df.loc[(df['lpa_name'] == 'Barnet') & (code == 'HSE'), 'category'].value_counts(normalize=True).round(2).head(5)

category
householder      0.99
full_planning    0.01
Name: proportion, dtype: float64

##### Task decisions
- **Job:** predict the category (legal route) from the description, to fill the ~31% of applications without one (mostly 2019–2020, before the form field existed).
- **Training data:** distinct descriptions from labelled applications, each with its most common category (majority vote); 'other' (123 rows) excluded. Split 80/10/10 at random: 409,961 / 51,245 / 51,246.
- **Second test set (missing_test):** 90,215 unlabelled applications from 18 authorities, labelled by council reference codes that mean one category at least 95% of the time across 30+ labelled examples. It measures the real job, including the shift in period and category mix.
- **Ceiling:** identical descriptions carry different categories, so a text-only model can't exceed about 91% accuracy. The classifier reports the most likely category with its probability.